GPT - Generatively pre-trained transformer

We are going to use Shakespear text to train transformers.

In [2]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

zsh:1: command not found: wget


In [3]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x1073490c0>)

In [4]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [6]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


# Tokenization
Below we do  simplest tokenization - character level tokenization
Google uses Sentencepiece - https://github.com/google/sentencepiece
OpenAi user tiktoken that uses BPE - https://github.com/openai/tiktoken

# Original BPE
compression or encodign  technique to encode longer strigns into shorter string using a translation table
Encode most common continuous sequesnce characters in a string with unsued 'placeholder' bytes. The iteration ends when no sequences can be found, leaving the target text effectively compressed

Example
aaabdaaabac
The byte pair "aa" occurs most often, so it will be replaced by a byte that is not used in the data, such as "Z"
```ZabdZabac
Z=aa```
Then the process is repeated with byte pair "ab", replacing it with "Y":
```
ZYdZYac
Y=ab
Z=aa
```

## Modified BPE
odified BPE does not aim to maximally compress text, but rather, to encode plaintext into "tokens", which are natural numbers
- All the unique tokens found in a corpus are listed in a token vocabulary. The token vocabulary can also include some other special tokens, relative to the use case
- vocabulary, in the case of GPT-3.5 and GPT-4, is 100258 (100000 from BPE algorithm and 258 included as special tokens)

Example:
Suppose we are encoding the previous example of "aaabdaaabac", with a specified vocabulary size of 6, then it would first be encoded as "0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3". Then it would proceed as before, and obtain "4, 5, 2, 4, 5, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5".

So far this is essentially the same as before. However, if we only had specified a vocabulary size of 5, then the process would stop at vocabulary "a=0, b=1, d=2, c=3, aa=4", so that the example would be encoded as "4, 0, 1, 2, 4, 0, 1, 0, 3". Conversely, if we had specified a vocabulary size of 8, then it would be encoded as "7, 6, 0, 3", with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5, aaab=6, aaabd=7". This is not maximally compressed, because modified BPE does not aim for maximum compression. Instead, it aims for an encoding that is efficient and practical for language model training


In [8]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

In [10]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

we don't train on entire dataset but only on chunks, why?

Why We Train on Chunks Instead of the Entire Dataset
There are 3 main reasons:
1. GPU Memory Constraints
The Shakespeare dataset has 1,115,394 characters
Training on the full sequence at once would require computing attention between ALL tokens simultaneously
Transformer attention has O(n²) memory complexity — for 1M tokens, that's ~1 trillion attention computations
This would exceed GPU memory limits instantly
2. Computational Efficiency
By using a fixed block_size (e.g., 8 or 256), we process manageable chunks
Each chunk becomes a training example with a predictable memory footprint
We can batch multiple chunks together for parallel processing
3. Better Learning Signal (Multiple Training Examples)
From a single sequence of length n, we get many overlapping training examples
With `block_size = 8`, the sequence `[18, 47, 56, 57, 58, 1, 15, 47, 58]` gives us:
```python
#Context → Target[18]                    → 47[18, 47]                → 56  [18, 47, 56]            → 57[18, 47, 56, 57]        → 58[18, 47, 56, 57, 58]    → 1# ... and so on
```
This trains the model to predict the next token given contexts of varying lengths (1 to block_size), making the model robust at inference time when it needs to handle different context sizes.

**In Summary**
Training on chunks is a practical necessity (memory/compute) that also provides a pedagogical benefit (varied context lengths for better generalization).

In [ ]:
block_size = 8
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

# Why block_size + 1?
When you see `train_data[:block_size+1]`, the +1 is needed because we need both inputs (x) and targets (y) from the same chunk.

Index:    0    1    2    3    4    5    6    7    8
Data:    [18,  47,  56,  57,  58,   1,  15,  47,  58]
          ↑_________________________↑    ↑
              x (inputs, 0-7)           y (last target)

In [12]:

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is {target}")
    


when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [ ]:
# above is time dimension of the dataset 
# we intorduce batch dimention fo the dataset so that batches are run in parallel on the gpu
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # batch_size number of random offsets 
    # ix is going to be a tensor of shape (batch_size,)
    # it's going to contain random integers between 0 and len(data) - block_size
    # these integers will be the starting indices of the chunks that we're going to take from the data
    x = torch.stack([data[i:i+block_size] for i in ix]) # (B, T)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # (B, T)
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('--------------------------------')
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context} the target is {target}")
        
# 32 exampels are compltely random for transformers packed into a batch

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
--------------------------------
when input is tensor([24]) the target is 43
when input is tensor([24, 43]) the target is 58
when input is tensor([24, 43, 58]) the target is 5
when input is tensor([24, 43, 58,  5]) the target is 57
when input is tensor([24, 43, 58,  5, 57]) the target is 1
when input is tensor([24, 43, 58,  5, 57,  1]) the target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
when input is tensor([44]) the target is 53
when input is tensor([44, 53]) the target is 56
when input is tensor

## Bigram Language Model - Detailed Explanation

### What is a Bigram Model?

A **bigram model** is the simplest possible language model. It predicts the next token based **only on the current token** — no history, no context. The question it answers is: "Given token X, what's likely to come next?"

---

### Understanding the Embedding Table

```python
self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
```

This creates a **learnable lookup table** of shape `(65, 65)` since our `vocab_size = 65` (unique characters in Shakespeare).

| Dimension | Meaning |
|-----------|---------|
| **Rows (65)** | One row for each possible input token (0-64) |
| **Columns (65)** | Logits (unnormalized scores) for each possible next token |

**Think of it as:** A 65×65 matrix where row `i` contains the model's predictions for "what comes after token `i`"

---

### The Forward Pass

```python
def forward(self, idx, targets):
    logits = self.token_embedding_table(idx)  # Output: (B, T, C)
```

**Input shapes:**
- `idx` = input tokens with shape `(Batch, Time)` e.g., `(4, 8)` means 4 sequences of 8 tokens each
- `targets` = target tokens (not used in this forward pass yet)

**What happens internally:**

For each token in `idx`, we "pluck out" that token's row from the embedding table:

```
idx = [[24, 43, 58, ...], ...]

Token 24 → Look up row 24 → Get 65 logits for "what comes after 24"
Token 43 → Look up row 43 → Get 65 logits for "what comes after 43"
Token 58 → Look up row 58 → Get 65 logits for "what comes after 58"
...
```

**Output:** `logits` with shape `(B, T, C)` = `(4, 8, 65)`
- B=4 batches
- T=8 time steps
- C=65 logit scores per position

---

### The Critical Limitation: No Context!

Each token makes its prediction **completely independently**:

```
Token "h" → predicts next (doesn't know it's in "hello")
Token "e" → predicts next (doesn't know "h" came before)
Token "l" → predicts next (doesn't know "he" came before)
```

**The tokens are NOT talking to each other.** They only see themselves — that's why this is called a "bigram" model (it only considers pairs: current → next).

---

### What are Logits?

Logits are **unnormalized log-probabilities**. To convert them to actual probabilities:

```python
probs = F.softmax(logits, dim=-1)  # Each row now sums to 1.0
```

---

### Why Start with Bigram?

This simple model is a stepping stone. Later, we'll add **self-attention** which allows tokens to "communicate" with each other and use context for much better predictions.


In [21]:
# now let's feed that into a simple bigram model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        # here we are creating a token embedding table of size vocab_size x vocab_size
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        # pluck out the row of embedding of the index
        # e.g. [24] it will pluck out 24th row of the embedding table
        
        logits = self.token_embedding_table(idx) # (Batch, Time/Sequence, Channel)
        # and so what's happening here is we are predicting what comes next based on just the individual identity of a single
        # token in the sequence, currently the tokens are not talking to each other and they're not
        # seeing any context except for they're just seeing themselves so I'm a f I'm a token number 24 and then I can
        # see what the logits are for the next token so the logits are the unnormalized probabilities of the next token

        # negative log likelihood loss
        # loss = F.cross_entropy(logits, targets) # this will not work because the logits are multi-dimensional
        # by default pytorch cross_entropy function want C to be the last dimension of the tensor
        # our logist are multi-dimensional wso we ned to convert to 2-dmensional to use cross entropy
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C) # B*T -> stretch the logit index to 1-d sequence and preseve the channel
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    # we need to generate a new token based on the context of the previous tokens
    def generate(self, idx, max_new_tokens):
        # this fucntion takes (B, T) array of indices in the current context in a batch and generate (B, T+1, +2, ....)
        # idx is (B, T) array of indices in the current context in a batch
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx) # (B, T, C)
            # focus only on the last time step, because thtese are the logits for the next token
            logits = logits[:, -1, :] # (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=-1) # (B, T+1)
        return idx
            
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) # we are expecting -ln(1/65) = 4.17
# idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))





torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [22]:
# train the model
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [28]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())
    


2.516516923904419


In [29]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))



Wherrowintherinche's d t nd alf d'sss:

Bu, we t ssthinon t, anthard wo amy, thir lame not
CAne g go


# Self-attention block

# Self-attention math trick


In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channel
x = torch.randn(B, T, C)

# let's see what the shape of x is
print(x.shape)

In [ ]:


# right now tokens are not talking to each other,
# we like to them to talk to each other
# information should flow from previous token to current token
# e.g. if at 5th token we want the context of 4,3,2,1,

# simplest way to preserve teh prededing token is takign the average of the previous tokens
# ie. we like to take channel at current token and also the channels of previous tokens 
# average them up and then use that feature vector as a context for the current token.
# averga is weak form of context, we need to do better

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t, :] = torch.mean(xprev, dim=0)
print(xbow.shape)

# the trick is dooing above in matrix multiplication
# let's look at simple matrix multiplication
torch.manual_seed(42)
a = torch.ones(3, 3) # 2 x 3 matrix
print('a', a)
b = torch.randint(0, 10, (3, 2)).float() # 3 x 2 matrix
print('b', b)
print('a @ b', a @ b)

# verstion 2
# the trick here is to use torch.tril to mask the upper triangular part of the matrix
a = a * torch.tril(torch.ones(3, 3))
print('a', a)
print('a @ b', a @ b)
# now `a @b` will have rows that are sum of previous rows
# by normalising the tril we can get the average
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, dim=1, keepdim=True)
print('a', a)
print('a @ b', a @ b)

# now let's vectorise the double for loop
wei = torch.tril(torch.ones(T, T)) # weights 
wei = wei / wei.sum(dim=1, keepdim=True)
print('wei', wei)
xbow2 = wei @ x # (4, 8, 8) @ (4, 8, 2) -> (4, 8, 2)
print('xbow2', xbow2)
torch.allclose(xbow, xbow2) # they are the same

# above is weighted aggregation 

# version 3
# let's do another version usign softmax
wei = torch.zeros((T, T))
wei = wei.masked_fill(torch.tril(torch.ones(T, T)), float('-inf')) # make the upper triangular part of the matrix -inf
wei = F.softmax(wei, dim=1) # softwmax is also a  normalisation operation that takes exponent and divides by sum
xbow3 = wei @ x 

# wei = torch.zeros((T, T))
# this v3 is better because think of wei above  as like an interaction matrix or affinity 
# which is telling us how much each token from the past do we want to aggregate and averahe io 

# this like `wei = wei.masked_fill(torch.tril(torch.ones(T, T)), float('-inf'))` is saying tokens from the future should not be considered

# Summary 
# short from this entire section is that you can do weighted aggregations of your past
# Elements by having by using matrix multiplication of a lower triangular
# fashion and then the elements here in the lower triangular part are telling you how much of each element uh fuses into this position


torch.Size([4, 8, 2])
a tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
a @ b tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])
a tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
a @ b tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])
a tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
a @ b tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
wei tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.

True

## Weighted Aggregation: The Mathematical Foundation of Self-Attention

This section demonstrates a **fundamental building block** of Transformers: how tokens can "communicate" with past tokens using matrix multiplication. This is the precursor to self-attention.

---

### The Problem: Tokens in Isolation

In our Bigram model, each token predicts the next token **without any context**. Token at position 5 doesn't know what happened at positions 1, 2, 3, 4.

**Goal:** We want information to flow from previous tokens to the current token.

---

### Solution: Weighted Aggregation of Past Tokens

The simplest way to give a token "context" is to **average** all the tokens that came before it.

---

## Version 1: Naive For-Loop Implementation

```python
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]           # All tokens from 0 to t (inclusive)
        xbow[b, t, :] = torch.mean(xprev, dim=0)  # Average them
```

For each position `t`, we take the mean of all tokens from position 0 to t:

| Position t | Tokens Averaged | Result |
|------------|-----------------|--------|
| 0 | x[0] | x[0] |
| 1 | x[0], x[1] | (x[0] + x[1]) / 2 |
| 2 | x[0], x[1], x[2] | (x[0] + x[1] + x[2]) / 3 |
| ... | ... | ... |

**Problem:** This nested loop is slow! O(B × T × T) operations.

---

## Version 2: Matrix Multiplication with Lower Triangular Matrix

### The Key Insight

Matrix multiplication can compute **weighted sums** in parallel!

Consider a simple example with 3 tokens:

```
a = [[1, 0, 0],      b = [[2, 7],
     [1, 1, 0],           [6, 4],
     [1, 1, 1]]           [6, 5]]
```

When we compute `a @ b`:

```
Row 0: 1×[2,7] + 0×[6,4] + 0×[6,5] = [2, 7]      ← Only token 0
Row 1: 1×[2,7] + 1×[6,4] + 0×[6,5] = [8, 11]    ← Sum of tokens 0,1
Row 2: 1×[2,7] + 1×[6,4] + 1×[6,5] = [14, 16]   ← Sum of tokens 0,1,2
```

### The Lower Triangular Matrix (tril)

```python
torch.tril(torch.ones(3, 3))
```
produces:
```
[[1, 0, 0],
 [1, 1, 0],
 [1, 1, 1]]
```

This **masks out the future** — each row only "sees" current and past positions.

### Normalizing to Get Averages

To get averages instead of sums, divide each row by its sum:

```python
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(dim=1, keepdim=True)
```
produces:
```
[[1.0000, 0.0000, 0.0000],   ← 1/1
 [0.5000, 0.5000, 0.0000],   ← 1/2, 1/2
 [0.3333, 0.3333, 0.3333]]   ← 1/3, 1/3, 1/3
```

Now `a @ b` computes **running averages** in one matrix multiply!

---

## The Mathematical Formulation

For a sequence of T tokens with C channels, the weighted aggregation is:

$$\text{xbow}[t] = \sum_{i=0}^{t} w_{t,i} \cdot x[i]$$

where $w_{t,i}$ are the weights (and $\sum_i w_{t,i} = 1$ for averaging).

In matrix form:
$$\text{xbow} = W \cdot X$$

where $W$ is a lower triangular weight matrix of shape $(T, T)$ and $X$ is $(T, C)$.

---

## Version 3: Using Softmax (The Attention Pattern!)

```python
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # Mask future positions
wei = F.softmax(wei, dim=-1)                      # Normalize
xbow3 = wei @ x
```

### Why Softmax?

1. **Exponential + Normalize:** `softmax(z)_i = exp(z_i) / Σexp(z_j)`
2. **Masked positions:** `-inf` → `exp(-inf) = 0` → completely ignored
3. **Unmasked positions:** `0` → `exp(0) = 1` → equal weight after normalization

### The Affinity Matrix

The `wei` matrix is an **affinity matrix** — it defines how much each past token contributes to the current token's representation:

```
wei = [[1.00, 0.00, 0.00, 0.00, ...],   ← Token 0 sees only itself
       [0.50, 0.50, 0.00, 0.00, ...],   ← Token 1 sees tokens 0,1 equally
       [0.33, 0.33, 0.33, 0.00, ...],   ← Token 2 sees tokens 0,1,2 equally
       ...]
```

**This is exactly what self-attention does, but with LEARNED weights instead of uniform ones!**

---

## From Uniform Weights to Self-Attention

| Version | Weight Matrix | Description |
|---------|---------------|-------------|
| **V1-V3 (here)** | Fixed uniform weights | Every past token contributes equally |
| **Self-Attention** | Learned data-dependent weights | Tokens learn which past tokens are relevant |

In self-attention, the weights come from:
$$w_{t,i} = \text{softmax}\left(\frac{Q_t \cdot K_i^T}{\sqrt{d_k}}\right)$$

where Q (Query) and K (Key) are learned projections of the tokens.

---

## Why is This Important for Language Models?

### Causal Masking

The lower triangular structure enforces **causality**:
- Token at position `t` can ONLY see tokens at positions `0, 1, ..., t`
- It CANNOT see future tokens `t+1, t+2, ...`
- This is essential for autoregressive language models that predict the next token

### Parallelization

Unlike RNNs that process sequentially, this matrix multiplication approach:
- Computes all positions in parallel
- Leverages GPU matrix operations
- Enables efficient training on long sequences

---

## Visual Summary

```
Input x:     [x₀]  [x₁]  [x₂]  [x₃]  ...
              ↓     ↓     ↓     ↓
              ┌─────┴─────┴─────┴─────┐
              │   Lower Triangular    │
              │    Weight Matrix W    │
              └─────┬─────┬─────┬─────┘
                    ↓     ↓     ↓
Output xbow: [x₀] [avg(x₀,x₁)] [avg(x₀,x₁,x₂)] ...

Each output position is a weighted combination of 
all previous inputs (including itself).
```

---

## Key Takeaways

1. **Matrix multiplication can compute weighted aggregations** of sequences efficiently
2. **Lower triangular matrices** enforce causality (no peeking at the future)
3. **Softmax normalization** ensures weights sum to 1 and handles masking gracefully
4. This is the **foundation of self-attention** — the only difference is that attention learns the weights instead of using uniform averages
5. The weight matrix `wei` will become the **attention scores** in the full Transformer


## Attention Variations Across Language Models

The weighted aggregation mechanism we just learned is the foundation of **attention**. Different language models have evolved various attention patterns to balance performance, efficiency, and context length. Here's a comprehensive overview:

---

## 1. Standard Self-Attention (Transformer - 2017)

**Used in:** Original Transformer, BERT, GPT-1/2

```
Attention(Q, K, V) = softmax(QK^T / √d_k) × V
```

### Masking Patterns

| Model Type | Mask | Can See |
|------------|------|---------|
| **Encoder (BERT)** | Bidirectional (no mask) | All tokens in sequence |
| **Decoder (GPT)** | Causal (lower triangular) | Only past tokens |
| **Encoder-Decoder (T5)** | Cross-attention | Encoder sees all; Decoder is causal |

### Complexity
- **Time:** O(n²) — every token attends to every other token
- **Memory:** O(n²) — must store full attention matrix
- **Limitation:** Doesn't scale well beyond ~2K-4K tokens

---

## 2. Multi-Head Attention

**Used in:** All modern Transformers (GPT, BERT, LLaMA, etc.)

Instead of one attention, run **h parallel attention heads** with smaller dimensions:

```python
# Instead of one 512-dim attention:
head_dim = 512 // 8  # = 64
heads = [Attention(Q_i, K_i, V_i) for i in range(8)]
output = Concat(heads) @ W_o
```

### Why Multiple Heads?

Each head can learn **different attention patterns**:
- Head 1: Focus on syntax (subject-verb agreement)
- Head 2: Focus on nearby tokens (local context)
- Head 3: Focus on semantic relationships
- Head 4: Focus on positional patterns

### Visualization

```
                    ┌─── Head 1: "The [cat] sat" ───┐
                    │                               │
Input ─────────────►├─── Head 2: "[The] cat [sat]" ─┼──► Concat ──► Output
"The cat sat"       │                               │
                    └─── Head 3: "The cat [sat]" ───┘
```

---

## 3. Sparse Attention Patterns

**Problem:** O(n²) is too expensive for long sequences.

**Solution:** Only compute attention for a subset of positions.

### 3a. Sliding Window Attention (Longformer, 2020)

**Used in:** Longformer, LED, BigBird

Each token only attends to **w neighbors** on each side:

```
Window size = 3

Token 5 attends to: [2, 3, 4, 5, 6, 7, 8]
                         ↑ current ↑
                    w=3 left    w=3 right
```

**Complexity:** O(n × w) — linear in sequence length!

### 3b. Dilated Sliding Window

**Used in:** Longformer (higher layers)

Like sliding window but with gaps (dilation):

```
Dilation = 2, Window = 3

Token 6 attends to: [0, 2, 4, 6, 8, 10, 12]
                              ↑
                         (every 2nd token)
```

Captures **longer-range dependencies** with same cost.

### 3c. Global + Local Attention (BigBird, 2020)

**Used in:** BigBird, Longformer

Some tokens (e.g., [CLS], special tokens) attend to **all positions**:

```
┌─────────────────────────────────────┐
│ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■  ← [CLS] sees all (global)
│ ■ ■ ■ □ □ □ □ □ □ □ □ □  ← Token 1 (local window)
│ ■ ■ ■ ■ □ □ □ □ □ □ □ □  ← Token 2 (local window)
│ ■ □ ■ ■ ■ □ □ □ □ □ □ □  ← Token 3 (local + global)
│ ...
└─────────────────────────────────────┘
■ = attends, □ = masked
```

---

## 4. Linear Attention Variants

**Problem:** Softmax prevents efficient computation.

### 4a. Performer (2020)

**Used in:** Performer

Approximates softmax with random features:

```python
# Standard: softmax(QK^T) @ V  →  O(n²)
# Performer: φ(Q) @ (φ(K)^T @ V)  →  O(n)
```

Uses **FAVOR+** (Fast Attention Via positive Orthogonal Random features).

### 4b. Linear Transformer (2020)

Replace softmax with simple feature maps:

```python
# φ(x) = elu(x) + 1
attention = (φ(Q) @ φ(K)^T) @ V
```

---

## 5. Grouped Query Attention (GQA)

**Used in:** LLaMA 2, Mistral, Falcon

**Problem:** KV cache grows linearly with sequence length during inference.

**Solution:** Share K and V heads across multiple Q heads:

```
Standard MHA (8 heads):     GQA (8 Q, 2 KV groups):
Q: 8 heads                  Q: 8 heads
K: 8 heads                  K: 2 heads (shared by 4 Q each)  
V: 8 heads                  V: 2 heads (shared by 4 Q each)

Memory: 8 + 8 + 8 = 24      Memory: 8 + 2 + 2 = 12
```

### Variants

| Type | Q Heads | KV Heads | Memory |
|------|---------|----------|--------|
| **MHA** | n | n | High |
| **GQA** | n | n/k | Medium |
| **MQA** (Multi-Query) | n | 1 | Lowest |

---

## 6. Flash Attention (2022)

**Used in:** LLaMA 2, GPT-4, Claude, Mistral, most modern LLMs

**Not a new attention pattern** — same math, but **memory-efficient implementation**:

```
Standard:
1. Compute full QK^T matrix (n × n) → store in HBM
2. Apply softmax
3. Multiply by V

Flash Attention:
1. Tile Q, K, V into blocks
2. Compute attention block-by-block in SRAM (fast)
3. Never materialize full n × n matrix
```

### Benefits

| Metric | Standard | Flash Attention |
|--------|----------|-----------------|
| Memory | O(n²) | O(n) |
| Speed | 1x | 2-4x faster |
| Exact? | Yes | Yes (not approximate!) |

---

## 7. Rotary Position Embeddings (RoPE)

**Used in:** LLaMA, GPT-NeoX, PaLM, Mistral, most modern LLMs

Instead of adding position embeddings, **rotate** the query and key vectors:

```python
# Position m encodes as rotation in 2D subspaces
R_m = [[cos(mθ), -sin(mθ)],
       [sin(mθ),  cos(mθ)]]

q_rotated = R_m @ q
k_rotated = R_n @ k

# Attention score depends on relative position (m - n)
score = q_rotated · k_rotated = f(m - n)
```

### Why RoPE?

1. **Relative positions:** Score depends on distance, not absolute position
2. **Extrapolation:** Can extend to longer sequences than training
3. **Efficiency:** No extra parameters, just rotation

---

## 8. Sliding Window + Attention Sinks (StreamingLLM, 2023)

**Used in:** Streaming inference, very long contexts

**Discovery:** First few tokens ("attention sinks") receive disproportionate attention.

```
Keep: [0, 1, 2, 3] + [n-w, ..., n-1, n]
       ↑ sinks ↑     ↑ sliding window ↑
```

Enables **infinite context** during inference!

---

## 9. Mixture of Experts (MoE) with Attention

**Used in:** Mixtral, GPT-4 (rumored), Switch Transformer

Not attention itself, but often combined:

```
                    ┌─── Expert 1 ───┐
                    │                │
Input ──► Router ──►├─── Expert 2 ───┼──► Weighted Sum ──► Output
                    │                │
                    └─── Expert 8 ───┘
                    
Only top-k experts activated per token (e.g., k=2)
```

---

## Comparison Table: Attention in Popular Models

| Model | Attention Type | Context | KV Sharing | Position |
|-------|---------------|---------|------------|----------|
| **GPT-2** | Causal MHA | 1024 | None | Learned |
| **GPT-3** | Causal MHA | 2048 | None | Learned |
| **GPT-4** | Causal + MoE? | 8K-128K | Unknown | Unknown |
| **BERT** | Bidirectional MHA | 512 | None | Learned |
| **LLaMA 1** | Causal MHA | 2048 | None | RoPE |
| **LLaMA 2** | Causal GQA | 4096 | GQA | RoPE |
| **Mistral 7B** | Sliding Window GQA | 8K (32K effective) | GQA | RoPE |
| **Mixtral** | Sliding + MoE | 32K | GQA | RoPE |
| **Longformer** | Global + Local | 4096+ | None | Learned |
| **Claude** | Causal + Flash | 100K+ | Unknown | Unknown |

---

## Evolution Timeline

```
2017: Transformer (Vaswani)
      └─► Standard self-attention, O(n²)

2018: GPT-1, BERT
      └─► Causal vs Bidirectional masking

2019: Transformer-XL
      └─► Segment-level recurrence for longer context

2020: Longformer, BigBird, Performer
      └─► Sparse attention, O(n) complexity

2021: RoPE (Su et al.)
      └─► Rotary embeddings for relative positions

2022: Flash Attention (Dao et al.)
      └─► Memory-efficient exact attention

2023: LLaMA 2, Mistral
      └─► GQA + RoPE + Flash Attention

2023: StreamingLLM
      └─► Attention sinks for infinite context

2024: Ring Attention, Striped Attention
      └─► Distributed attention across devices
```

---

## Key Insights

1. **The core math is the same** — softmax(QK^T/√d) × V — but implementations vary wildly

2. **Memory is the bottleneck**, not compute — Flash Attention proves this

3. **Sparse patterns work** — you don't need full n² attention for good performance

4. **KV caching matters** — GQA/MQA reduce inference memory by 4-8x

5. **Position encoding is crucial** — RoPE enables length extrapolation

6. **Modern LLMs combine multiple techniques:**
   - Flash Attention (memory)
   - GQA (KV cache)
   - RoPE (positions)
   - Sliding window (long context)


# Self attention
```python
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # Mask future positions
wei = F.softmax(wei, dim=-1)                      # Normalize
xbow3 = wei @ x
```
with above aech token gathers informaton through averaging but we don't want uniform information , we want it to be data dependent. For exampel if I'm a vowel then maybe I'm looking for consonants in my past and maybe I want toand I want that information to flow to me and so I want to now gather information from the past but I want to do it in the data dependent way and this is the problem that self attention solves

Every single token at its postion emits 2 vectors.
1. Q - "what i m looking for?"
2. k - "what do i contain"
dot product Q.K gives the affinity between tokens 

In [ ]:
# self attention head implmentaion 
torch.manual_seed(1337)
B, T, C = 4, 8, 32 # batch, time, channel
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)

tril = torch.tril(torch.ones(T, T))
wei = q @ k.transpose(-2, -1) #(B,T,16) @ (B, 16, T) -> (B, T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x) # (B, T, head_size) # this is the value that we want to attend to for this head
out = wei @ v
out.shape




# we want to compute the attention weights for each token
# we want to compute the attention weights for each token


torch.Size([4, 8, 16])

In [59]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [ ]:
wei[0]
# each token now knows waht position it is in the sequence and what content it has
# based on that it creates a query e.g. "hey I'm looking for this kind of stuff um I'm a vowel I'm on the 8th position I'm looking for any consonant at positions upto 4"
#  all the nodes get to emit keys and maybe one of the channels
 # maybe one of the channels
# could be I am a I am a consonant and I am in a position up to four and that that key would have a high number in
# that specific Channel and that's how the query and the key when they do product they can find each other and create a
# high affinity and when they have a high Affinity like say uh this token was
# pretty interesting to uh to this eighth token

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)


Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
# Why scaled attention
# k, q are unit variance but dot prodcut is not 
# this is then fed to softmax and that causes unfair diffusion of weights
# with very high or low input to softmax converges it to one-hot vectors
# so we scale it by 1/sqrt(head_size)
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
# wei = q @ k.transpose(-2, -1) # (B, T, T)
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # scaling



In [64]:
k.var()

tensor(0.9006)

In [65]:
q.var()

tensor(1.0037)

In [66]:
wei.var()

tensor(15.9317)

In [ ]:
# show the impac of high values in softmax
# softmax will converge towars the the highest value 
# means it ndoes will get information from one ndoe
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [68]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

## Why Scaled Attention? A Deep Dive

### The Problem: Variance Explosion in Dot Products

When we compute attention scores via `Q @ K.T`, something mathematically dangerous happens:

**If Q and K have unit variance (var ≈ 1), their dot product does NOT have unit variance!**

---

### Mathematical Proof

Let's say we have two random vectors `q` and `k`, each with `d` dimensions (where `d = head_size`):

```
q = [q₁, q₂, ..., qd]   where each qᵢ ~ N(0, 1)
k = [k₁, k₂, ..., kd]   where each kᵢ ~ N(0, 1)
```

The dot product is:
```
q · k = q₁k₁ + q₂k₂ + ... + qdkd
```

Each term `qᵢkᵢ` has:
- Mean: E[qᵢkᵢ] = E[qᵢ]E[kᵢ] = 0 × 0 = 0
- Variance: Var(qᵢkᵢ) = E[qᵢ²]E[kᵢ²] = 1 × 1 = 1

Since we're summing `d` independent terms:
```
Var(q · k) = d × 1 = d
```

**The variance of the dot product grows linearly with dimension `d`!**

For `head_size = 64`, the dot product has variance ≈ 64, meaning values can be 8× larger than expected.

---

### Why is High Variance Bad for Softmax?

Softmax is defined as:
```
softmax(xᵢ) = exp(xᵢ) / Σⱼ exp(xⱼ)
```

**Problem:** Softmax is extremely sensitive to the scale of its inputs!

| Input Values | Softmax Behavior |
|--------------|------------------|
| Small (around 0) | Produces diffuse/uniform distribution |
| Large (positive/negative) | Converges to one-hot vector |

---

### Visual Demonstration

**Case 1: Small input values (good)**
```python
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)
# Output: tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
```
→ Weights are **diffuse** — token can gather information from multiple sources.

**Case 2: Large input values (bad)**
```python
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]) * 8, dim=-1)
# Output: tensor([0.0326, 0.0066, 0.0882, 0.0066, 0.8659])
```
→ Weights **collapse to one-hot** — token only sees ONE other token!

---

### The Gradient Problem

When softmax saturates (approaches one-hot):
- Gradients become extremely small (vanishing gradients)
- The model can't learn from most positions
- Training becomes unstable

```
Attention weights:  [0.001, 0.001, 0.001, 0.997]
                      ↓       ↓       ↓       ↓
Gradients:         [tiny,  tiny,  tiny,  okay]
```

Only the "winner" position gets meaningful gradient updates!

---

### The Solution: Scale by 1/√d

To restore unit variance, we divide by √d:

```python
wei = (q @ k.transpose(-2, -1)) * (head_size ** -0.5)
```

**Why √d?**

If `Var(q·k) = d`, then:
```
Var((q·k) / √d) = Var(q·k) / d = d / d = 1
```

The attention scores now have unit variance, keeping softmax in its "well-behaved" regime.

---

### Empirical Verification

```python
head_size = 16
k = torch.randn(B, T, head_size)  # var ≈ 1
q = torch.randn(B, T, head_size)  # var ≈ 1

# Without scaling
wei_unscaled = q @ k.transpose(-2, -1)
print(f"Unscaled variance: {wei_unscaled.var():.2f}")  # ≈ 16 (= head_size)

# With scaling
wei_scaled = wei_unscaled * (head_size ** -0.5)
print(f"Scaled variance: {wei_scaled.var():.2f}")      # ≈ 1
```

---

### Summary: The Scaled Dot-Product Attention Formula

The complete attention formula from "Attention Is All You Need":

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- **Q, K, V**: Query, Key, Value matrices
- **dₖ**: Dimension of key vectors (head_size)
- **1/√dₖ**: The critical scaling factor

---

### Key Takeaways

1. **Dot products amplify variance** — grows linearly with dimension
2. **High variance breaks softmax** — causes one-hot saturation
3. **One-hot attention is bad** — tokens only "see" one other token
4. **Scaling by 1/√d fixes this** — restores unit variance
5. **Diffuse attention is better for learning** — allows gradient flow to all positions


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # residual connection
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4196, val loss 2.4337
step 400: train loss 2.3500, val loss 2.3562
step 500: train loss 2.2964, val loss 2.3126
step 600: train loss 2.2405, val loss 2.2497
step 700: train loss 2.2050, val loss 2.2180
step 800: train loss 2.1636, val loss 2.1863
step 900: train loss 2.1235, val loss 2.1489
step 1000: train loss 2.1028, val loss 2.1295
step 1100: train loss 2.0706, val loss 2.1193
step 1200: train loss 2.0376, val loss 2.0786
step 1300: train loss 2.0261, val loss 2.0652
step 1400: train loss 1.9920, val loss 2.0366
step 1500: train loss 1.9696, val loss 2.0304
step 1600: train loss 1.9590, val loss 2.0437
step 1700: train loss 1.9406, val loss 2.0127
step 1800: train loss 1.9089, val loss 1.9962
step 1900: train loss 1.9083, val loss 1.9873
step 2000: train loss 1.8848, val loss 1.9950
step 2100: train loss 1.

# Transformer has followign blocks 

the block basically intersperses communication and then computation the computation the communication is done using multi-headed selfelf attention and then the computation is done using a feed forward.

1. self-attention block (communication among tokens)
2. fed-farward (computation, i.e when ndoes get data from other nodes what eo do wiht it)

this is all optimised then uising Residual connections and layer norm

# The Transformer Block: A Complete Architectural Deep Dive

The Transformer architecture from "Attention Is All You Need" (Vaswani et al., 2017) is built from repeated **blocks** that interleave **communication** and **computation**. Let's dissect each component using our implementation from the previous cell.

---

## The Original Transformer Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                      TRANSFORMER ARCHITECTURE                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  Input: "The cat sat"                                           │
│         ↓                                                        │
│  ┌──────────────────┐                                           │
│  │ Token Embedding  │  "The" → [0.1, -0.3, 0.8, ...]           │
│  │       +          │                                            │
│  │ Position Embed   │  pos_0 → [0.2, 0.1, -0.1, ...]           │
│  └────────┬─────────┘                                           │
│           ↓                                                      │
│  ╔════════════════════════════════════════╗                     │
│  ║         TRANSFORMER BLOCK (×N)         ║  ← Repeated N times │
│  ╠════════════════════════════════════════╣                     │
│  ║                                        ║                     │
│  ║   ┌────────────────────────────┐      ║                     │
│  ║   │      Layer Norm 1          │      ║                     │
│  ║   └─────────────┬──────────────┘      ║                     │
│  ║                 ↓                      ║                     │
│  ║   ┌────────────────────────────┐      ║                     │
│  ║   │  Multi-Head Self-Attention │      ║  ← COMMUNICATION    │
│  ║   │     (tokens talk)          │      ║                     │
│  ║   └─────────────┬──────────────┘      ║                     │
│  ║                 ↓                      ║                     │
│  ║            (+) ← Residual Connection   ║                     │
│  ║                 ↓                      ║                     │
│  ║   ┌────────────────────────────┐      ║                     │
│  ║   │      Layer Norm 2          │      ║                     │
│  ║   └─────────────┬──────────────┘      ║                     │
│  ║                 ↓                      ║                     │
│  ║   ┌────────────────────────────┐      ║                     │
│  ║   │     Feed-Forward Network   │      ║  ← COMPUTATION      │
│  ║   │     (tokens think)         │      ║                     │
│  ║   └─────────────┬──────────────┘      ║                     │
│  ║                 ↓                      ║                     │
│  ║            (+) ← Residual Connection   ║                     │
│  ║                 ↓                      ║                     │
│  ╚════════════════════════════════════════╝                     │
│           ↓                                                      │
│  ┌──────────────────┐                                           │
│  │ Final Layer Norm │                                           │
│  └────────┬─────────┘                                           │
│           ↓                                                      │
│  ┌──────────────────┐                                           │
│  │   Linear Head    │  → Logits for next token                  │
│  └──────────────────┘                                           │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

---

## 1. The Block: Communication + Computation

The **Block** is the fundamental repeating unit. Each block does two things:

| Phase | Component | Purpose | Analogy |
|-------|-----------|---------|---------|
| **Communication** | Multi-Head Self-Attention | Tokens exchange information | "Let's discuss what we know" |
| **Computation** | Feed-Forward Network | Each token processes its information | "Let me think about this" |

### Our Implementation (from previous cell)

```python
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)  # Communication
        self.ffwd = FeedFoward(n_embd)                   # Computation
        self.ln1 = nn.LayerNorm(n_embd)                  # Pre-norm 1
        self.ln2 = nn.LayerNorm(n_embd)                  # Pre-norm 2

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # Residual + Attention
        x = x + self.ffwd(self.ln2(x)) # Residual + FFN
        return x
```

### Data Flow Through One Block

```
Input x: (B, T, 64)
    │
    ├───────────────────────────────┐
    │                               │
    ↓                               │
┌───────────────┐                   │
│  LayerNorm1   │                   │
└───────┬───────┘                   │
        ↓                           │
┌───────────────────────┐           │
│ Multi-Head Attention  │           │ Residual
│   (4 heads × 16 dim)  │           │ Connection
└───────┬───────────────┘           │
        ↓                           │
        + ←─────────────────────────┘
        │
        ├───────────────────────────┐
        │                           │
        ↓                           │
┌───────────────┐                   │
│  LayerNorm2   │                   │
└───────┬───────┘                   │
        ↓                           │
┌───────────────────────┐           │ Residual
│  Feed-Forward Network │           │ Connection
│   (64 → 256 → 64)     │           │
└───────┬───────────────┘           │
        ↓                           │
        + ←─────────────────────────┘
        │
        ↓
Output: (B, T, 64)
```

---

## 2. Multi-Head Self-Attention (Communication)

This is where tokens **talk to each other**. Instead of one attention mechanism, we run multiple "heads" in parallel, each looking for different patterns.

### Why Multiple Heads?

```
Single Head (64 dim):
  - Can only learn ONE attention pattern
  - Either syntax OR semantics, not both

Multi-Head (4 heads × 16 dim = 64 dim):
  - Head 1: "Which nouns relate to which verbs?"
  - Head 2: "What's the nearby context?"
  - Head 3: "What's the subject of this sentence?"
  - Head 4: "Are there any negations before me?"
```

### Our Implementation

```python
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)  # Output projection
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads in parallel, then concatenate
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, n_embd)
        out = self.dropout(self.proj(out))  # Project back + regularize
        return out
```

### Single Attention Head

```python
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)    # What do I contain?
        self.query = nn.Linear(n_embd, head_size, bias=False)  # What am I looking for?
        self.value = nn.Linear(n_embd, head_size, bias=False)  # What do I communicate?
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        
        # Compute attention scores with scaling
        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, T)
        
        # Causal mask: prevent attending to future tokens
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)  # Normalize to probabilities
        wei = self.dropout(wei)
        
        # Weighted aggregation of values
        v = self.value(x)  # (B, T, head_size)
        out = wei @ v      # (B, T, head_size)
        return out
```

### Attention Computation Visualized

```
Query (what am I looking for?):
Token 5: "I need verb information"  →  q₅ = [0.8, -0.2, 0.5, ...]

Key (what do I contain?):
Token 2: "I am a verb"              →  k₂ = [0.9, -0.1, 0.4, ...]
Token 3: "I am an adjective"        →  k₃ = [0.1, 0.8, -0.3, ...]

Affinity (dot product):
q₅ · k₂ = 0.8×0.9 + (-0.2)×(-0.1) + 0.5×0.4 = 0.94  ← HIGH! Similar
q₅ · k₃ = 0.8×0.1 + (-0.2)×0.8 + 0.5×(-0.3) = -0.23 ← LOW! Different

After softmax, token 5 will attend strongly to token 2 (the verb)!
```

---

## 3. Feed-Forward Network (Computation)

After tokens have gathered information from each other, each token **independently** processes what it learned. This is where "thinking" happens.

### Our Implementation

```python
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # Expand: 64 → 256
            nn.ReLU(),                       # Non-linearity
            nn.Linear(4 * n_embd, n_embd),  # Contract: 256 → 64
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)
```

### Why 4× Expansion?

```
Input:  64 dimensions (compressed representation)
        ↓
Hidden: 256 dimensions (expanded "thinking space")
        ↓
Output: 64 dimensions (compressed back)
```

The expansion to 4× gives the network more capacity to learn complex transformations. It's like giving each token a larger "scratch pad" to work with before summarizing back.

### Key Point: Position-wise

The FFN operates **independently** on each position:

```
Position 0: x₀ → FFN → y₀
Position 1: x₁ → FFN → y₁  (same FFN weights, but no cross-talk)
Position 2: x₂ → FFN → y₂
...
```

This is in contrast to attention, where all positions interact.

---

## 4. Residual Connections (Skip Connections)

The `+` operations in our block are **residual connections**:

```python
x = x + self.sa(self.ln1(x))   # Add attention output to input
x = x + self.ffwd(self.ln2(x)) # Add FFN output to input
```

### Why Residual Connections?

```
Without Residual:                With Residual:
x → [Attention] → y              x → [Attention] → y
                                      ↓
Gradient must flow through      x + y (gradient has direct path!)
entire attention block
```

**Benefits:**
1. **Gradient Highway**: Gradients can flow directly through the `+` during backprop
2. **Easier Optimization**: Network can learn identity function by setting layer weights to zero
3. **Deeper Networks**: Enables training of very deep models (100+ layers)

### The Gradient Flow

```
Forward:  x → f(x) → x + f(x)
Backward: ∂L/∂x = ∂L/∂(x+f(x)) × (1 + ∂f/∂x)
                                  ↑
                              Always at least 1!
```

The gradient is **at least 1** even if the layer learns nothing. This prevents vanishing gradients.

---

## 5. Layer Normalization (Stability)

LayerNorm normalizes activations to have mean=0 and variance=1 across the embedding dimension.

```python
self.ln1 = nn.LayerNorm(n_embd)  # Normalizes the 64 embedding dimensions
```

### What LayerNorm Does

```
Input: x = [2.0, -1.0, 3.0, 0.5, ...]  (64 values, any scale)
        ↓
Mean:   μ = mean(x) = 1.125
Std:    σ = std(x) = 1.5
        ↓
Output: (x - μ) / σ = [0.58, -1.42, 1.25, -0.42, ...]  (mean≈0, var≈1)
        ↓
Scale:  γ × output + β  (learned scale and shift)
```

### Pre-Norm vs Post-Norm

**Original Transformer (Post-Norm):**
```python
x = LayerNorm(x + Attention(x))
x = LayerNorm(x + FFN(x))
```

**Our Implementation (Pre-Norm):**
```python
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

**Pre-Norm is preferred** because:
- More stable training
- Better gradient flow
- Used in GPT-2, GPT-3, LLaMA, etc.

---

## 6. The Full Model: Stacking Blocks

Our complete model stacks multiple blocks together:

```python
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # Embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)     # 65 → 64
        self.position_embedding_table = nn.Embedding(block_size, n_embd)  # 32 → 64
        
        # Stack of transformer blocks
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        
        # Final processing
        self.ln_f = nn.LayerNorm(n_embd)              # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)  # 64 → 65 (logits)
```

### Forward Pass Walkthrough

```python
def forward(self, idx, targets=None):
    B, T = idx.shape  # (16, 32)

    # Step 1: Embed tokens and positions
    tok_emb = self.token_embedding_table(idx)  # (16, 32, 64)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (32, 64)
    x = tok_emb + pos_emb  # (16, 32, 64) - broadcasts position to all batches

    # Step 2: Pass through all transformer blocks
    x = self.blocks(x)  # (16, 32, 64) - 4 blocks of attention + FFN

    # Step 3: Final normalization
    x = self.ln_f(x)  # (16, 32, 64)

    # Step 4: Project to vocabulary
    logits = self.lm_head(x)  # (16, 32, 65) - one logit per vocab token
    
    return logits, loss
```

---

## 7. Complete Architecture Summary

| Component | Shape Transformation | Parameters | Purpose |
|-----------|---------------------|------------|---------|
| Token Embedding | (B,T) → (B,T,64) | 65 × 64 = 4,160 | Map tokens to vectors |
| Position Embedding | (T,) → (T,64) | 32 × 64 = 2,048 | Encode position info |
| Block (×4) | (B,T,64) → (B,T,64) | ~50K each | Communication + Computation |
| ├─ LayerNorm1 | (B,T,64) → (B,T,64) | 128 | Normalize before attention |
| ├─ MultiHead Attn | (B,T,64) → (B,T,64) | ~16K | 4 attention heads |
| ├─ LayerNorm2 | (B,T,64) → (B,T,64) | 128 | Normalize before FFN |
| └─ FeedForward | (B,T,64) → (B,T,64) | ~33K | MLP computation |
| Final LayerNorm | (B,T,64) → (B,T,64) | 128 | Final normalization |
| LM Head | (B,T,64) → (B,T,65) | 64 × 65 = 4,160 | Project to vocabulary |
| **Total** | | **~210K** | |

---

## 8. The Intuition: Why This Works

```
┌─────────────────────────────────────────────────────────────┐
│                     THE TRANSFORMER RECIPE                   │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Block 1:  "Hey tokens, share what you know"                │
│            (Multi-Head Attention)                            │
│            "Now each of you, think about what you heard"    │
│            (Feed-Forward)                                    │
│                                                              │
│  Block 2:  "Share again with your new understanding"        │
│            (Multi-Head Attention)                            │
│            "Think deeper"                                    │
│            (Feed-Forward)                                    │
│                                                              │
│  Block 3:  "Now you have higher-level insights, share"      │
│            (Multi-Head Attention)                            │
│            "Almost there, process this"                      │
│            (Feed-Forward)                                    │
│                                                              │
│  Block 4:  "Final coordination"                             │
│            (Multi-Head Attention)                            │
│            "Make your prediction"                            │
│            (Feed-Forward)                                    │
│                                                              │
│  Output:   "Given everything, predict the next token"       │
│            (LM Head)                                         │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

---

## 9. Mapping to the Original Paper

Our implementation corresponds to the **Decoder** stack in "Attention Is All You Need":

```
Paper Terminology          Our Implementation
─────────────────────────────────────────────────────
N = 6 layers               n_layer = 4 blocks
d_model = 512              n_embd = 64
h = 8 heads                n_head = 4
d_k = d_v = 64             head_size = 16
d_ff = 2048                4 * n_embd = 256
```

We use a smaller model for fast training on CPU, but the architecture is identical!

---

## Key Takeaways

1. **Block = Communication + Computation**
   - Attention lets tokens talk
   - FFN lets tokens think

2. **Multi-Head = Multiple Perspectives**
   - Each head learns different patterns
   - Heads run in parallel, then concatenate

3. **Residual Connections = Gradient Highways**
   - Enable training of deep networks
   - Information can skip layers

4. **Layer Normalization = Stability**
   - Keeps activations well-behaved
   - Pre-norm is more stable than post-norm

5. **Stacking = Hierarchical Understanding**
   - Early layers: local patterns, syntax
   - Later layers: global patterns, semantics

6. **Position Embeddings = Spatial Awareness**
   - Attention has no inherent notion of order
   - Must explicitly encode position
